In [11]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/features_v1.csv")
df.head()

,qualifyingPosition,pitStopCount,driver_form_avg,constructor_form_avg,circuitType_street,finishPosition,avgLapTime_s,constructorPoints
0,NaN,0.0,1.8,NaN,False,13,NaN,0.0
1,NaN,0.0,1.8,NaN,False,13,NaN,0.0
2,NaN,0.0,1.8,NaN,False,13,NaN,0.0
3,NaN,0.0,1.8,NaN,False,13,NaN,0.0
4,NaN,0.0,1.8,NaN,False,13,NaN,0.0


In [12]:
target = "constructorPoints"

features = [
    "qualifyingPosition",
    "pitStopCount",
    "driver_form_avg",
    "constructor_form_avg",
    "circuitType_street"
]

reuse engineered features for fair comparison across tasks and cleaner project narrative

In [13]:
df_cp = df.dropna(subset=features + [target]).copy()

df_cp[target].describe()

count    331196.000000
mean          7.037398
std          10.449135
min           0.000000
25%           0.000000
50%           2.000000
75%          10.000000
max          66.000000
Name: constructorPoints, dtype: float64

In [14]:
X_cp = df_cp[features]
y_cp = df_cp[target]

In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_cp, y_cp, test_size=0.2, random_state=42
)

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

lr_cp = LinearRegression()
lr_cp.fit(X_train, y_train)

y_pred_lr = lr_cp.predict(X_test)

print("Constructor Points — Linear Regression")
print("MAE:", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R²:", r2_score(y_test, y_pred_lr))

Constructor Points — Linear Regression
MAE: 3.7081393437423333
RMSE: 5.703615948108825
R²: 0.7050656322159119


In [17]:
from sklearn.ensemble import RandomForestRegressor

rf_cp = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

rf_cp.fit(X_train, y_train)

y_pred_rf = rf_cp.predict(X_test)

print("Constructor Points — Random Forest")
print("MAE:", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R²:", r2_score(y_test, y_pred_rf))

Constructor Points — Random Forest
MAE: 0.2072316297812206
RMSE: 0.8171370317839823
R²: 0.9939463874889388


In [19]:
df_cp[["constructorPoints", "constructor_form_avg"]].corr()

,constructorPoints,constructor_form_avg
constructorPoints,1.000000,0.823382
constructor_form_avg,0.823382,1.000000


In [20]:
importance = pd.Series(
    rf_cp.feature_importances_,
    index=features
).sort_values(ascending=False)

importance

constructor_form_avg    0.743338
driver_form_avg         0.110378
qualifyingPosition      0.085828
pitStopCount            0.050723
circuitType_street      0.009733
dtype: float64

Constructor form emerged as the most important feature, confirming that recent team performance strongly influences race outcomes. Qualifying position and driver form also contributed meaningful predictive signal, while pit stop count had a smaller effect.